# 04 — Tranche pricing under three credit models

**Project:** *Selling Property Rental Ownership — A Hybrid Real Estate Model*  
**Author:** Dan Allouche · **Date:** May 2026

We re-run the simulation under each of the three credit models (Gaussian one-factor, Student-t one-factor, Cox doubly stochastic) and compare the resulting fair prices, loss distributions and tail behaviour. This is the empirical evidence motivating the post-2008 critique of the Gaussian copula.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from tranche_pricing.config import load_config
from tranche_pricing.pricing import compare_credit_models, runner as pricing_runner
from tranche_pricing.viz import figures
from tranche_pricing.viz.style import apply_style

apply_style()
pd.options.display.float_format = "{:,.4f}".format
cfg = load_config(ROOT / "config/paris_intermediate.yaml")
sim, gbm, vas, lgd, tranches, coupons = pricing_runner.build_inputs_from_yaml(cfg)


## Side-by-side comparison across the three credit models

In [ ]:
df, outputs = compare_credit_models(sim, gbm=gbm, vasicek=vas, lgd=lgd, tranches=tranches, coupons=coupons)
display(df.set_index(["credit_model", "instrument"])[["fair_price", "fair_to_par", "mean_ann_return", "risk_std", "risk_var_95", "risk_es_95"]])


## Figure #4 — Loss distribution overlay

In [ ]:
losses_by_model = {name: out.cumulative_loss[:, -1] for name, out in outputs.items()}
fig = figures.fig_loss_distributions(losses_by_model=losses_by_model)
fig.savefig(ROOT / "artifacts/figures/fig_loss_distributions.pdf")
fig.savefig(ROOT / "artifacts/figures/fig_loss_distributions.png", dpi=200)
for name, losses in losses_by_model.items():
    print(f"{name:24s} mean={losses.mean():.3f}  VaR95={np.quantile(losses, 0.95):.3f}  VaR99={np.quantile(losses, 0.99):.3f}")


## Figure #5 — Lower tail dependence under the Student-t copula

In [ ]:
fig = figures.fig_tail_dependence()
fig.savefig(ROOT / "artifacts/figures/fig_tail_dependence.pdf")
fig.savefig(ROOT / "artifacts/figures/fig_tail_dependence.png", dpi=200)


---
**Next step:** `05_stress_and_sensitivity.ipynb` replays GFC / COVID / rate-hike scenarios and surfaces the parameter sensitivities.